# Buffalo Embedding Comparison

This notebook is only for identity embedding checks using Buffalo (w600k_r50 via InsightFace).

Use it for:
- image vs image similarity
- video frame consistency
- video vs image similarity
- video vs reference-folder similarity

In [6]:
from pathlib import Path
import random
import json

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from insightface.app import FaceAnalysis
from insightface.utils import face_align

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("cuda available:", torch.cuda.is_available())

device: cuda
cuda available: True


In [7]:
# -----------------------------
# Config
# -----------------------------
INSIGHTFACE_MODEL_NAME = "buffalo_l"
INSIGHTFACE_ROOT = None
DET_SIZE = (640, 640)
FACE_SIZE = 224
RANDOM_SEED = 42
MAX_VIDEO_FRAMES = 12
USE_IMAGE_FALLBACK_IF_NO_FACE = True

IMAGE_PATH_A = Path(r"C:\DSP\Crops\20260309_002737_frame32_face0.jpg")
IMAGE_PATH_B = Path(r"C:\DSP\Crops\20260309_002737_frame151_face0.jpg")
VIDEO_PATH = Path(r"C:\DSP\video.mp4")
REFERENCE_IMAGE_DIR = Path(r"C:\DSP\reference_images")

print("image A:", IMAGE_PATH_A)
print("image B:", IMAGE_PATH_B)
print("video:", VIDEO_PATH)
print("reference dir:", REFERENCE_IMAGE_DIR)

image A: C:\DSP\Crops\20260309_002737_frame32_face0.jpg
image B: C:\DSP\Crops\20260309_002737_frame151_face0.jpg
video: C:\DSP\video.mp4
reference dir: C:\DSP\reference_images


In [8]:
# -----------------------------
# Load Buffalo
# -----------------------------
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device.type == "cuda" else ['CPUExecutionProvider']
face_kwargs = {"name": INSIGHTFACE_MODEL_NAME, "providers": providers}
if INSIGHTFACE_ROOT:
    face_kwargs["root"] = INSIGHTFACE_ROOT

face_app = FaceAnalysis(**face_kwargs)
face_app.prepare(ctx_id=0 if device.type == "cuda" else -1, det_size=DET_SIZE)
print("Buffalo loaded")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [9]:
# -----------------------------
# Helpers
# -----------------------------
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def pick_largest_face(faces):
    if not faces:
        return None
    return sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)[0]


def get_aligned_face(frame_bgr, face_obj, out_size=224):
    if hasattr(face_obj, "kps") and face_obj.kps is not None:
        aligned_bgr = face_align.norm_crop(frame_bgr, landmark=face_obj.kps, image_size=out_size)
        aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(aligned_rgb)

    x1, y1, x2, y2 = face_obj.bbox.astype(int)
    h, w = frame_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    crop_bgr = cv2.resize(crop_bgr, (out_size, out_size))
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(crop_rgb)


def normalize_embedding(embedding):
    embedding = np.asarray(embedding, dtype=np.float32)
    return embedding / np.linalg.norm(embedding)


def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def detect_face_with_padding(frame_bgr, padding_fracs=(0.0, 0.25, 0.5, 1.0)):
    h, w = frame_bgr.shape[:2]
    for frac in padding_fracs:
        pad_y = int(h * frac)
        pad_x = int(w * frac)
        if pad_x == 0 and pad_y == 0:
            padded = frame_bgr
        else:
            padded = cv2.copyMakeBorder(
                frame_bgr,
                pad_y,
                pad_y,
                pad_x,
                pad_x,
                borderType=cv2.BORDER_REFLECT101,
            )
        faces = face_app.get(padded)
        face = pick_largest_face(faces)
        if face is not None:
            return padded, face, frac
    return None, None, None


def load_face_and_embedding_from_image(image_path):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)

    frame_bgr = cv2.imread(str(image_path))
    if frame_bgr is None:
        raise RuntimeError(f"Could not read image: {image_path}")

    detect_frame, face, used_padding = detect_face_with_padding(frame_bgr)
    if face is None:
        raise RuntimeError(f"No face detected in image: {image_path}")

    if used_padding and used_padding > 0:
        print(f"Used padded detection fallback ({used_padding:.2f}) for {image_path.name}")

    aligned = get_aligned_face(detect_frame, face, out_size=FACE_SIZE)
    if aligned is None:
        raise RuntimeError(f"Could not align face from image: {image_path}")

    embedding = normalize_embedding(face.embedding)
    return {
        "path": str(image_path),
        "aligned_face": aligned,
        "embedding": embedding,
    }


def sample_video_faces_and_embeddings(video_path, max_frames=12, seed=42):
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise RuntimeError("Could not determine total frame count")

    sample_count = min(max_frames, total_frames)
    rng = random.Random(seed)
    frame_indices = sorted(rng.sample(range(total_frames), sample_count))

    records = []
    progress = tqdm(frame_indices, desc="sample video", dynamic_ncols=True)
    for frame_idx in progress:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame_bgr = cap.read()
        if not ok:
            continue
        faces = face_app.get(frame_bgr)
        face = pick_largest_face(faces)
        if face is None:
            continue
        aligned = get_aligned_face(frame_bgr, face, out_size=FACE_SIZE)
        if aligned is None:
            continue
        records.append({
            "frame_index": frame_idx,
            "aligned_face": aligned,
            "embedding": normalize_embedding(face.embedding),
        })
        progress.set_postfix({"faces": len(records)})

    cap.release()
    return records


def load_reference_embeddings(reference_dir):
    reference_dir = Path(reference_dir)
    if not reference_dir.exists():
        raise FileNotFoundError(reference_dir)

    image_paths = sorted([p for p in reference_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])
    if not image_paths:
        raise RuntimeError(f"No reference images found in {reference_dir}")

    records = []
    progress = tqdm(image_paths, desc="reference embeddings", dynamic_ncols=True)
    for image_path in progress:
        try:
            record = load_face_and_embedding_from_image(image_path)
            records.append(record)
            progress.set_postfix({"kept": len(records)})
        except Exception as exc:
            print(f"Skipping {image_path.name}: {exc}")

    if not records:
        raise RuntimeError("No usable reference faces found")

    embeddings = np.stack([record["embedding"] for record in records], axis=0)
    mean_embedding = embeddings.mean(axis=0)
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    return {
        "records": records,
        "embeddings": embeddings,
        "mean_embedding": mean_embedding,
    }


def summarize_video_consistency(video_records):
    if len(video_records) < 2:
        raise RuntimeError(f"Need at least 2 detected faces from the video, got {len(video_records)}")

    embeddings = np.stack([record["embedding"] for record in video_records], axis=0)
    matrix = embeddings @ embeddings.T
    off_diag = matrix[~np.eye(matrix.shape[0], dtype=bool)]
    mean_embedding = embeddings.mean(axis=0)
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    to_mean = embeddings @ mean_embedding

    print("sampled frames:", [record["frame_index"] for record in video_records])
    print("num embeddings:", len(video_records))
    print("pairwise cosine mean:", f"{off_diag.mean():.4f}")
    print("pairwise cosine min :", f"{off_diag.min():.4f}")
    print("pairwise cosine max :", f"{off_diag.max():.4f}")
    print("similarity to mean  :", [round(float(x), 4) for x in to_mean.tolist()])
    print("pairwise matrix:\n", matrix)

    return {
        "matrix": matrix,
        "embeddings": embeddings,
        "mean_embedding": mean_embedding,
        "to_mean": to_mean,
    }

In [10]:
# -----------------------------
# Image vs image comparison
# -----------------------------
if IMAGE_PATH_A.exists() and IMAGE_PATH_B.exists():
    rec_a = load_face_and_embedding_from_image(IMAGE_PATH_A)
    rec_b = load_face_and_embedding_from_image(IMAGE_PATH_B)
    sim_ab = cosine_similarity(rec_a["embedding"], rec_b["embedding"])

    print("image A:", rec_a["path"])
    print("image B:", rec_b["path"])
    print("buffalo embedding similarity:", f"{sim_ab:.4f}")
else:
    print("Set IMAGE_PATH_A and IMAGE_PATH_B to valid files before running this cell.")

Used padded detection fallback (0.25) for 20260309_002737_frame32_face0.jpg
Used padded detection fallback (0.25) for 20260309_002737_frame151_face0.jpg
image A: C:\DSP\Crops\20260309_002737_frame32_face0.jpg
image B: C:\DSP\Crops\20260309_002737_frame151_face0.jpg
buffalo embedding similarity: 0.5037


In [ ]:
# -----------------------------
# Video consistency check
# -----------------------------
video_consistency = None
if VIDEO_PATH.exists():
    video_records = sample_video_faces_and_embeddings(VIDEO_PATH, max_frames=MAX_VIDEO_FRAMES, seed=RANDOM_SEED)
    video_consistency = summarize_video_consistency(video_records)
else:
    print("Set VIDEO_PATH to a valid file before running this cell.")

In [ ]:
# -----------------------------
# Video vs image comparison
# -----------------------------
if VIDEO_PATH.exists() and IMAGE_PATH_A.exists():
    if video_consistency is None:
        video_records = sample_video_faces_and_embeddings(VIDEO_PATH, max_frames=MAX_VIDEO_FRAMES, seed=RANDOM_SEED)
        video_consistency = summarize_video_consistency(video_records)

    rec_a = load_face_and_embedding_from_image(IMAGE_PATH_A)
    sim_video_to_image = cosine_similarity(video_consistency["mean_embedding"], rec_a["embedding"])

    print("video:", VIDEO_PATH)
    print("image:", rec_a["path"])
    print("video mean vs image similarity:", f"{sim_video_to_image:.4f}")
else:
    print("Set VIDEO_PATH and IMAGE_PATH_A to valid files before running this cell.")

In [ ]:
# -----------------------------
# Video vs reference folder comparison
# -----------------------------
if VIDEO_PATH.exists() and REFERENCE_IMAGE_DIR.exists():
    if video_consistency is None:
        video_records = sample_video_faces_and_embeddings(VIDEO_PATH, max_frames=MAX_VIDEO_FRAMES, seed=RANDOM_SEED)
        video_consistency = summarize_video_consistency(video_records)

    reference_profile = load_reference_embeddings(REFERENCE_IMAGE_DIR)
    gallery_scores = reference_profile["embeddings"] @ video_consistency["mean_embedding"]
    best_score = float(gallery_scores.max())
    mean_score = float(np.dot(reference_profile["mean_embedding"], video_consistency["mean_embedding"]))

    print("reference images used:", len(reference_profile["records"]))
    print("best video vs reference score:", f"{best_score:.4f}")
    print("video mean vs reference mean:", f"{mean_score:.4f}")
    print("all gallery scores:", [round(float(x), 4) for x in gallery_scores.tolist()])
else:
    print("Set VIDEO_PATH and REFERENCE_IMAGE_DIR to valid paths before running this cell.")